In [ ]:
!pip install -q transformers accelerate torch sentencepiece peft
!pip install -q datasets evaluate nltk sqlparse rouge-score codebert-score PyYAML matplotlib pandas

In [ ]:
import os, sys
%matplotlib inline
import json
import logging
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("checkpoint_2")
from src.config_loader import CFG
from src.dataset_loader import UnifiedDatasetLoader
from src.models.tokenizer import CodeTokenizer
from src.models.codegen_loader_small_model import CodeGenModelWrapper
from src.generators.doc_generator import generate_docs_batch, pairs_from_codocbench
from src.evaluation.metrics import CodeMetricsEvaluator
from src.generators.sql_generator import evaluate_batch as evaluate_sql_batch
from src.generators.program_generator import synthesize_batch
from src.evaluation.comparator import ArchitectureComparator
from src.evaluation.visualizer import ResultsVisualizer
from IPython.display import Image, display

In [ ]:
REPO_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    # e.g. running on Colab: clone the repo if it isn't already checked out
    if not os.path.isdir("Emasters_Group-2_CapstoneProject"):
        !git clone -q https://github.com/Rithusravya/Emasters_Group-2_CapstoneProject.git
    REPO_ROOT = os.path.abspath("Emasters_Group-2_CapstoneProject")
    os.chdir(REPO_ROOT)

sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

## 1. Load config, datasets, and the base model

In [ ]:
atasets = UnifiedDatasetLoader.load_datasets(CFG)
for name, rows in datasets.items():
    sources = sorted({r.get("_source_file") for r in rows if isinstance(r, dict)})
    print(f"{name:12s} -> {len(rows):4d} rows  (from {sources})")

In [ ]:
model = CodeGenModelWrapper(
    model_name=CFG.model.name_or_path,
    device=CFG.project.device,
    max_length=CFG.model.max_length,
    use_lora=True,
    lora_kwargs=dict(CFG.lora),
)
print("Loaded:", CFG.model.name_or_path, "on", model.device)

In [ ]:
N_DOC_SAMPLES = 20
N_SQL_SAMPLES = 20
N_PROGRAM_SAMPLES = 10

## 2. Documentation generation — BERTScore

In [ ]:
doc_pairs = pairs_from_codocbench(datasets["codocbench"])[:N_DOC_SAMPLES]
doc_codes, doc_references = (list(x) for x in zip(*doc_pairs)) if doc_pairs else ([], [])

doc_predictions = generate_docs_batch(model, doc_codes) if doc_codes else []
doc_metrics = CodeMetricsEvaluator.evaluate_generation(doc_predictions, doc_references, lang="python")

print(f"Doc generation — n={doc_metrics['n']}")
print(f"  BLEU           : {doc_metrics['bleu']:.4f}")
print(f"  ROUGE-L        : {doc_metrics['rouge_l']:.4f}")
print(f"  CodeBLEU       : {doc_metrics['codebleu']:.4f}")
print(f"  BERTScore (F1) : {doc_metrics['code_bertscore']:.4f}")
print(f"  Exact match    : {doc_metrics['exact_match']:.4f}")

if doc_predictions:
    print("\n--- sample ---")
    print("code       :", doc_codes[0][:120].replace(chr(10), ' '))
    print("reference  :", doc_references[0][:160])
    print("prediction :", doc_predictions[0][:160])

## 3. Text-to-SQL generation

In [ ]:
sql_examples = [
    {
        "schema": f"-- schema for db '{row.get('db_id', 'unknown')}' not bundled locally; "
                  f"infer table/column names from the question.",
        "question": row["question"],
        "gold_sql": row["SQL"],
        "db_path": None,
    }
    for row in datasets["birdbench"][:N_SQL_SAMPLES]
    if row.get("question") and row.get("SQL")
]

sql_result = evaluate_sql_batch(model, sql_examples)
print(f"Text-to-SQL — n={sql_result['n']}")
print(f"  Exact-match accuracy : {sql_result['exact_match_acc']:.4f}")
print(f"  Execution accuracy   : {sql_result['execution_acc']:.4f}  (0.0 expected — no local DBs)")

if sql_result["predictions"]:
    print("\n--- sample ---")
    print("question :", sql_examples[0]["question"][:160])
    print("gold     :", sql_examples[0]["gold_sql"][:160])
    print("predicted:", sql_result["predictions"][0][:160])

## 4. Program generation

In [ ]:
program_problems = [
    "Return the maximum value in a list of integers.",
    "Check whether a string is a palindrome.",
    "Compute the nth Fibonacci number.",
][:N_PROGRAM_SAMPLES]

program_outputs = synthesize_batch(model, program_problems)
for problem, out in zip(program_problems, program_outputs):
    print(f"# {problem}\n{out}\n{'-'*60}")

In [ ]:
results = {
    "Documentation Generation": {
        "n": doc_metrics["n"],
        "Accuracy": doc_metrics["exact_match"],
        "BERTScore": doc_metrics["code_bertscore"],
        "BLEU": doc_metrics["bleu"],
        "ROUGE-L": doc_metrics["rouge_l"],
        "CodeBLEU": doc_metrics["codebleu"],
    },
    "Text-to-SQL Generation": {
        "n": sql_result["n"],
        "Accuracy": sql_result["exact_match_acc"],
        "BERTScore": None,  # not applicable — SQL correctness isn't a semantic-similarity task
        "ExecutionAccuracy": sql_result["execution_acc"],
    },
}

summary_df = pd.DataFrame(results).T
summary_df

## 5. Visualization

In [ ]:
tasks = list(results.keys())
accuracy_vals = [results[t]["Accuracy"] or 0.0 for t in tasks]
bertscore_vals = [results[t]["BERTScore"] if results[t]["BERTScore"] is not None else 0.0 for t in tasks]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(tasks, accuracy_vals, color="#1f77b4")
axes[0].set_title("Accuracy (exact match) by task")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.0)
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(accuracy_vals):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center")

axes[1].bar(tasks, bertscore_vals, color="#ff7f0e")
axes[1].set_title("BERTScore (F1) by task")
axes[1].set_ylabel("BERTScore F1")
axes[1].set_ylim(0, 1.0)
axes[1].tick_params(axis="x", rotation=15)
for i, v in enumerate(bertscore_vals):
    label = f"{v:.2f}" if results[tasks[i]]["BERTScore"] is not None else "n/a"
    axes[1].text(i, v + 0.02, label, ha="center")

plt.tight_layout()
plt.show()

## 6. Comparator

In [ ]:
architecture_comparison = ArchitectureComparator.compare_architectures()
chart_path = ResultsVisualizer.generate_chart(architecture_comparison, CFG.outputs.plots_dir,
                                               filename="checkpoint_2_architecture_comparison.png")
display(Image(filename=chart_path))

In [ ]:
os.makedirs(CFG.outputs.metrics_dir, exist_ok=True)
ResultsVisualizer.save_metrics(results, f"{CFG.outputs.metrics_dir}/checkpoint_2_results.json")

fig.savefig(os.path.join(CFG.outputs.plots_dir, "checkpoint_2_accuracy_bertscore.png"), dpi=300)

print("Saved:")
print(" -", f"{CFG.outputs.metrics_dir}/checkpoint_2_results.json")
print(" -", os.path.join(CFG.outputs.plots_dir, "checkpoint_2_accuracy_bertscore.png"))
print(" -", chart_path)